In [1]:
import numpy as np
import os 
import math

os.chdir('../../')
from load import load_bin_dir_to_ratio
from metric import calculate_correlation_with_groundtruth, calculate_precision_racall, calculate_dict_correlation
from binlib import output_bin

ref_genome_folder = 'Data/ref_genome/hg19'
exclude_chr = ['chrM', 'chrY', 'chrMT']
resolution = 50000

# load ref genome length info
chr_length = dict()
with open(ref_genome_folder + '/hg19.len', 'r') as f:
    lines = f.read().splitlines()
    for line in lines:
        chr_name , len_ = line.split('\t')[:2]
        if chr_name in exclude_chr:
            continue
        chr_length[chr_name] = int(len_)
print('chr length : \n',chr_length)

# calculate bin number in each chr
chr_bin_number = dict()
for key in chr_length.keys():
    chr_bin_number[key] = math.floor(chr_length[key] / resolution) + 1
print('chr bin number : \n', chr_bin_number)

# back to current dir
os.chdir('Exps/exp3_pool_residual')

chr length : 
 {'chr1': 249250621, 'chr2': 243199373, 'chr3': 198022430, 'chr4': 191154276, 'chr5': 180915260, 'chr6': 171115067, 'chr7': 159138663, 'chr8': 146364022, 'chr9': 141213431, 'chr10': 135534747, 'chr11': 135006516, 'chr12': 133851895, 'chr13': 115169878, 'chr14': 107349540, 'chr15': 102531392, 'chr16': 90354753, 'chr17': 81195210, 'chr18': 78077248, 'chr19': 59128983, 'chr20': 63025520, 'chr21': 48129895, 'chr22': 51304566, 'chrX': 155270560}
chr bin number : 
 {'chr1': 4986, 'chr2': 4864, 'chr3': 3961, 'chr4': 3824, 'chr5': 3619, 'chr6': 3423, 'chr7': 3183, 'chr8': 2928, 'chr9': 2825, 'chr10': 2711, 'chr11': 2701, 'chr12': 2678, 'chr13': 2304, 'chr14': 2147, 'chr15': 2051, 'chr16': 1808, 'chr17': 1624, 'chr18': 1562, 'chr19': 1183, 'chr20': 1261, 'chr21': 963, 'chr22': 1027, 'chrX': 3106}


In [ ]:
bin_dir = '../../CNV/K562_HIC_CNV/HiNTcnv_OUTPUT_K562_HindIII/segmentation/b50000'

res_dict = dict()
for key in chr_bin_number.keys():
    res_dict[key] = np.zeros(chr_bin_number[key])

keys = list(chr_bin_number.keys())
key_num = 0
for file in sorted(os.listdir(bin_dir)):
    print(file)
    with open(f'{bin_dir}/{file}', 'r') as f:
        for line in f.readlines()[1:]:
            start_, end_, obs, exp ,var = line.strip().split()
            residual = float(obs) - float(exp)
            bin_ = math.floor(int(start_)/resolution)
            res_dict[keys[key_num]][bin_] = residual
    key_num = key_num +1


In [2]:
win_size_list = [3,5,11,21,41]

### operate averge pooling in defferent window size

In [18]:

for win_size in win_size_list:
    pool_res_dict = dict()
    for key in chr_bin_number.keys():
        pool_res_dict[key] = np.zeros(chr_bin_number[key])


    for chr_ in res_dict.keys():
        for i in range(len(res_dict[chr_])):
            start_ = i - math.floor(win_size/2) if i - math.floor(win_size/2) >=0 else 0 
            end_ = i + math.floor(win_size/2) if i + math.floor(win_size/2) <= chr_bin_number[chr_] else chr_bin_number[chr_]
            win_ = res_dict[chr_][start_:end_+1]
            pool_res_dict[chr_][i] = np.mean(win_)

    #output .bw for visualization
    output_path = f'hic_residual_pool_win{win_size}.bw'
    with open(output_path, 'w+') as f:
        for key in pool_res_dict.keys():
            for bin_, residual in enumerate(pool_res_dict[key]):
                bin_start = bin_ * 50000
                bin_end = (bin_+1) * 50000 if (bin_+1) * 50000 <= chr_length[key] else chr_length[key]
                f.write(f'{key}\t{bin_start}\t{bin_end}\t{residual}\n')

    ### output for BICseq-seg
    output_bin(pool_res_dict, f'win{win_size}')

### calculate log2 residual ratio corr.

In [19]:
for win_size in win_size_list:
    print(f'win {win_size}')
    hic_log2_ratio_dict = load_bin_dir_to_ratio(f'win{win_size}')
    wgs_log2_ratio_dict = load_bin_dir_to_ratio('../../CNV/K562_WGS_CNV/bin')
    calculate_dict_correlation(hic_log2_ratio_dict, wgs_log2_ratio_dict)

win 3
Pearson corr. :  0.755
Spearman corr. :  0.827
win 5
Pearson corr. :  0.751
Spearman corr. :  0.832
win 11
Pearson corr. :  0.703
Spearman corr. :  0.837
win 21
Pearson corr. :  0.675
Spearman corr. :  0.845
win 41
Pearson corr. :  0.621
Spearman corr. :  0.84


### need to run BICseq-seg manually

(linux)
../../BICseq2-seg_v0.7.3/BICseq2-seg.pl win41/_config.txt win5/exp3_win41_lambda3.seg --title pooled_HiC_win41_lambda3 --fig win41/exp3_win41_lambda3.png --lambda 3 --bootstrap

### metrics

#### lambda : 3

In [3]:
for win_size in win_size_list:
    print(f'win size : {win_size}')
    calculate_correlation_with_groundtruth(f'win{win_size}/exp3_win{win_size}_lambda3.seg', groundtruth_file = "../../CNV/K562_WGS_CNV/K562_WGS_CNV_lambda4.txt")
    print('\n')

win size : 3
Pearson corr. :  0.708
Spearman corr. :  0.847


win size : 5
Pearson corr. :  0.745
Spearman corr. :  0.842


win size : 11
Pearson corr. :  0.713
Spearman corr. :  0.868


win size : 21
Pearson corr. :  0.675
Spearman corr. :  0.862


win size : 41
Pearson corr. :  0.645
Spearman corr. :  0.855




In [4]:

for win_size in win_size_list:
    print(f'win size : {win_size}')
    calculate_precision_racall(f'win{win_size}/exp3_win{win_size}_lambda3.seg', groundtruth_file = "../../CNV/K562_WGS_CNV/K562_WGS_CNV_lambda4.txt")
    print('\n')

win size : 3
Number of ground truth : 74
Number of target : 25
Number of hits : 20
precision : 0.8
recall : 0.27

hits by size : 
	 < 1Mb :  0
	1Mb - 2Mb :  1
	2Mb - 5Mb :  2
	5Mb - 10Mb :  1
	10Mb - 20Mb :  1
	20Mb - 50Mb :  8
	>= 50Mb :  7
sum of  hit size : 796388018
average hit fraction : 0.826


win size : 5
Number of ground truth : 74
Number of target : 28
Number of hits : 22
precision : 0.786
recall : 0.297

hits by size : 
	 < 1Mb :  0
	1Mb - 2Mb :  1
	2Mb - 5Mb :  3
	5Mb - 10Mb :  1
	10Mb - 20Mb :  1
	20Mb - 50Mb :  8
	>= 50Mb :  8
sum of  hit size : 873288016
average hit fraction : 0.84


win size : 11
Number of ground truth : 74
Number of target : 31
Number of hits : 24
precision : 0.774
recall : 0.324

hits by size : 
	 < 1Mb :  0
	1Mb - 2Mb :  1
	2Mb - 5Mb :  2
	5Mb - 10Mb :  1
	10Mb - 20Mb :  1
	20Mb - 50Mb :  11
	>= 50Mb :  8
sum of  hit size : 951767910
average hit fraction : 0.847


win size : 21
Number of ground truth : 74
Number of target : 32
Number of hits : 22
pre

#### lambda 2

In [5]:
for win_size in win_size_list:
    print(f'win size : {win_size}')
    calculate_correlation_with_groundtruth(f'win{win_size}/exp3_win{win_size}_lambda2.seg', groundtruth_file = "../../CNV/K562_WGS_CNV/K562_WGS_CNV_lambda4.txt")
    print('\n')

win size : 3
Pearson corr. :  0.835
Spearman corr. :  0.878


win size : 5
Pearson corr. :  0.863
Spearman corr. :  0.864


win size : 11
Pearson corr. :  0.804
Spearman corr. :  0.834


win size : 21
Pearson corr. :  0.743
Spearman corr. :  0.829


win size : 41
Pearson corr. :  0.633
Spearman corr. :  0.836




In [6]:
for win_size in win_size_list:
    print(f'win size : {win_size}')
    calculate_precision_racall(f'win{win_size}/exp3_win{win_size}_lambda2.seg', groundtruth_file = "../../CNV/K562_WGS_CNV/K562_WGS_CNV_lambda4.txt")
    print('\n')

win size : 3
Number of ground truth : 74
Number of target : 36
Number of hits : 29
precision : 0.806
recall : 0.392

hits by size : 
	 < 1Mb :  2
	1Mb - 2Mb :  2
	2Mb - 5Mb :  3
	5Mb - 10Mb :  2
	10Mb - 20Mb :  1
	20Mb - 50Mb :  11
	>= 50Mb :  8
sum of  hit size : 969388009
average hit fraction : 0.886


win size : 5
Number of ground truth : 74
Number of target : 37
Number of hits : 30
precision : 0.811
recall : 0.405

hits by size : 
	 < 1Mb :  2
	1Mb - 2Mb :  2
	2Mb - 5Mb :  4
	5Mb - 10Mb :  2
	10Mb - 20Mb :  1
	20Mb - 50Mb :  11
	>= 50Mb :  8
sum of  hit size : 971638008
average hit fraction : 0.863


win size : 11
Number of ground truth : 74
Number of target : 41
Number of hits : 29
precision : 0.707
recall : 0.392

hits by size : 
	 < 1Mb :  1
	1Mb - 2Mb :  2
	2Mb - 5Mb :  3
	5Mb - 10Mb :  2
	10Mb - 20Mb :  1
	20Mb - 50Mb :  12
	>= 50Mb :  8
sum of  hit size : 981417905
average hit fraction : 0.858


win size : 21
Number of ground truth : 74
Number of target : 42
Number of hits : 

#### lambda : 1

In [7]:
for win_size in win_size_list:
    print(f'win size : {win_size}')
    calculate_correlation_with_groundtruth(f'win{win_size}/exp3_win{win_size}_lambda1.seg', groundtruth_file = "../../CNV/K562_WGS_CNV/K562_WGS_CNV_lambda4.txt")
    print('\n')

win size : 3
Pearson corr. :  0.829
Spearman corr. :  0.868


win size : 5
Pearson corr. :  0.843
Spearman corr. :  0.864


win size : 11
Pearson corr. :  0.784
Spearman corr. :  0.858


win size : 21
Pearson corr. :  0.733
Spearman corr. :  0.842


win size : 41
Pearson corr. :  0.62
Spearman corr. :  0.82




In [8]:
for win_size in win_size_list:
    print(f'win size : {win_size}')
    calculate_precision_racall(f'win{win_size}/exp3_win{win_size}_lambda1.seg', groundtruth_file = "../../CNV/K562_WGS_CNV/K562_WGS_CNV_lambda4.txt")
    print('\n')

win size : 3
Number of ground truth : 74
Number of target : 49
Number of hits : 34
precision : 0.694
recall : 0.459

hits by size : 
	 < 1Mb :  3
	1Mb - 2Mb :  2
	2Mb - 5Mb :  4
	5Mb - 10Mb :  3
	10Mb - 20Mb :  3
	20Mb - 50Mb :  11
	>= 50Mb :  8
sum of  hit size : 1014088004
average hit fraction : 0.878


win size : 5
Number of ground truth : 74
Number of target : 53
Number of hits : 35
precision : 0.66
recall : 0.473

hits by size : 
	 < 1Mb :  2
	1Mb - 2Mb :  2
	2Mb - 5Mb :  4
	5Mb - 10Mb :  4
	10Mb - 20Mb :  2
	20Mb - 50Mb :  13
	>= 50Mb :  8
sum of  hit size : 1056088003
average hit fraction : 0.829


win size : 11
Number of ground truth : 74
Number of target : 62
Number of hits : 30
precision : 0.484
recall : 0.405

hits by size : 
	 < 1Mb :  1
	1Mb - 2Mb :  2
	2Mb - 5Mb :  3
	5Mb - 10Mb :  3
	10Mb - 20Mb :  2
	20Mb - 50Mb :  15
	>= 50Mb :  4
sum of  hit size : 854779240
average hit fraction : 0.81


win size : 21
Number of ground truth : 74
Number of target : 70
Number of hits : 